In [1]:
import pandas as pd
from cobra.io import read_sbml_model
import optlang
import optlang_enumerator.cobra_cnapy
import optlang_enumerator.mcs_computation as mcs_computation
import numpy as np
from pathlib import Path

In [2]:
#model_name = "M_model"
model_name = "PQS_model"

In [3]:
model = read_sbml_model('../models/' + model_name + ".xml")

In [4]:
conversions = pd.read_csv('../results/dual_' + model_name + '.csv')
total_conversions = conversions.shape[0]
conversions = conversions[conversions['M_MW'] != 0]
conversions_supp_w = conversions.shape[0]
#conversions

In [5]:
total_conversions

111

In [6]:
conversions_supp_w

17

In [7]:
conversions_copy = conversions.copy()

In [8]:
# Step 1: Set all positive values from irreversible reactions to zero
irrev = [not rxn.reversibility for rxn in model.reactions] + [True]
mask = np.array(irrev, dtype=bool)

conversions_copy.loc[:, mask] = conversions_copy.loc[:, mask].clip(upper=0)

# Step 2: For each row, get column names of nonzero (negative) values
nonzero_cols = [
    frozenset(conversions_copy.columns[row != 0])
    for _, row in conversions_copy.iterrows()
]

# Remove empty sets (rows that were all zero/positive)
nonzero_cols = [s for s in nonzero_cols if s]

# Step 3: Remove duplicates and supersets
# Keep a set only if no other set is a proper subset of it
unique_sets = list(set(nonzero_cols))  # remove exact duplicates first

minimal_sets = []
for s in unique_sets:
    # Keep s only if there is no other set that is a strict subset of s
    if not any(other < s for other in unique_sets if other != s):
        minimal_sets.append(s)

# Optional: convert back to sorted lists for readability
minimal_sets = sorted([sorted(s) for s in minimal_sets], key=len)

In [9]:
minimal_sets

[['M_R01'],
 ['M_R04'],
 ['M_R03', 'M_R05'],
 ['M_R05', 'M_R06'],
 ['M_R05', 'M_R10'],
 ['M_R10', 'M_R11'],
 ['M_R06', 'M_R07', 'M_R11'],
 ['M_R03', 'M_R07', 'M_R11']]

In [10]:
len(minimal_sets)

8

In [11]:
clean_sets = [{rxn.split('_')[1] for rxn in min_set} for min_set in minimal_sets]

In [12]:
#M_model
#target = 'r5'

#PQS
target = "R04"

In [13]:
results_cache_dir = None # do not cache preprocessing results
# results_cache_dir = Path(".") # cache preprocessing results (in the current directory)

In [14]:
mod = optlang_enumerator.cobra_cnapy.CNApyModel.read_sbml_model("../models/" + model_name + ".xml")
# allow all reactions that are not boundary reactions as cuts (same as exclude_boundary_reactions_as_cuts option of compute_mcs)
cuts = np.array([not not r for r in mod.reactions])
reac_id = mod.reactions.list_attr('id') # list of reaction IDs in the model
# define target (multiple targets are possible; each target can have multiple linear inequality constraints)
mod_mue_target = [[(target, ">=", 0.01)]] # one target with one constraint, a.k.a. syntehtic lethals
# this constraint alone would not be sufficient, but there are uptake limits defined in the reaction bounds
# of the model that are integerated below, first, however, parse the constraint(s) and convert to matrix format
mod_mue_target = [mcs_computation.relations2leq_matrix(
                   mcs_computation.parse_relations(t, reac_id_symbols=mcs_computation.get_reac_id_symbols(reac_id)), reac_id)
                   for t in mod_mue_target]
# now integrate the non-default reaction bounds from the network
mcs_computation.integrate_model_bounds(mod, mod_mue_target)
# this is just to show what the first (and only) target now looks like:
for c in mcs_computation.get_leq_constraints(mod, mod_mue_target)[0]:
    print(c, c.problem)

bab6fbd8-505d-11f1-8194-bc2411973405: -1.0*R04 + 1.0*R04_reverse_529de <= -0.01 None


In [15]:
# calculate MCS up to size 6 with "any MCS" enumeration method
mod_mcs,_ = mcs_computation.compute_mcs(mod, mod_mue_target, cuts=cuts, enum_method=3, max_mcs_size=6, network_compression=True,
                                         include_model_bounds=True, max_mcs_num=100000, timeout=6000, results_cache_dir=results_cache_dir)
print(len(mod_mcs), "MCS found.")
# show MCS as n-tuples of reaction IDs
mod_mcs_rxns= [tuple(reac_id[r] for r in mcs) for mcs in mod_mcs]

# check that all MCS disable the first (and only) target
print(all(mcs_computation.check_mcs(mod, mod_mue_target[0], mod_mcs, optlang.interface.INFEASIBLE)))

FVA to find blocked reactions...
Network compression...
Using indicators.
Objective function is empty; set objective to self.minimize_sum_over_z
Found solution with objective value 1.0
MCS ['R04']
Found solution with objective value 2.0
CS ['R05' 'R06'] -> MCS ['R05' 'R06']
Found solution with objective value 2.0
CS ['R10' 'R11'] -> MCS ['R10' 'R11']
Found solution with objective value 3.0
CS ['R01' 'R05' 'R11'] -> MCS ['R01']
Found solution with objective value 3.0
CS ['R03' 'R05' 'R11'] -> MCS ['R03' 'R05']
Found solution with objective value 3.0
CS ['R06' 'R07' 'R11'] -> MCS ['R06' 'R07' 'R11']
Found solution with objective value 2.0
MCS ['R05' 'R10']
Found solution with objective value 3.0
MCS ['R03' 'R07' 'R11']
Stopping enumeration with status infeasible
8 MCS found.
True


In [16]:
# Convert mod_mcs_rxns tuples to sets of frozensets for easy comparison
mod_set = {frozenset(rxns) for rxns in mod_mcs_rxns}

# Compare converted_mets (ignoring 'MW') against mod_mcs_rxns
matches = []
for entry in clean_sets:
    rxns = frozenset(x for x in entry if x != 'MW')
    if rxns in mod_set:
        matches.append(entry)

print(f"Number of matches: {len(matches)}")
print("Matched entries:")
for m in matches:
    print(m)

Number of matches: 8
Matched entries:
{'R01'}
{'R04'}
{'R03', 'R05'}
{'R06', 'R05'}
{'R10', 'R05'}
{'R11', 'R10'}
{'R11', 'R06', 'R07'}
{'R11', 'R03', 'R07'}
